# Combining the detected stance files

In [1]:
import pandas as pd

In [4]:
ifrom = 0
ito = 82054
istep = 500
scored_directory = "data/EU Debates/scored/"
dfs = [
    pd.read_csv(f"{scored_directory}/scored_data_llama_{i}_{min(i+istep, ito)}.csv", sep=";", encoding="utf-8-sig")
    for i in range(ifrom, ito, istep)
]

df = pd.concat(dfs, ignore_index=True)
'''
df["EU Party"] = df["EU Party"].apply(lambda x: x.split('/')[-1])
df = df[~df["EU Party"].str.contains(r"\bNA\b", case=False, na=False)]
df["EU Party"].value_counts()
'''
df.columns

Index(['text', 'score', 'reason', 'speaker_party'], dtype='object')

In [43]:
df.to_csv(f"{scored_directory}/full.csv", index=False)

# Now combine all the data

In [44]:
df_eudebates = pd.read_csv(f"{scored_directory}/full.csv")
df_europarl = pd.read_csv("data/EuroParl Custom/en_parties_classifier.csv")
df_europarl = df_europarl.dropna(subset=["en", "party_group_std"])

In [39]:
df_europarl.columns

Index(['Unnamed: 0', 'year', 'en', 'party_group_std'], dtype='object')

In [40]:
df_eudebates.columns

Index(['text', 'score', 'reason', 'speaker_party'], dtype='object')

In [47]:
df_europarl.rename(columns={"en":"text", "party_group_std":"speaker_party"}, inplace=True)


In [50]:
filtered_debates = df_eudebates[~df_eudebates['text'].isin(df_europarl['text'])]
filtered_debates = filtered_debates[filtered_debates["score"]>=0.6]

In [53]:
filtered_debates = filtered_debates[["text", "speaker_party"]]

In [63]:
df_dev = df_europarl[df_europarl["year"]==2010][["text","speaker_party"]]
df_dev.drop_duplicates(subset=["text"], inplace=True)
df_test = df_europarl[df_europarl["year"]==2011][["text","speaker_party"]]
df_dev.drop_duplicates(subset=["text"], inplace=True)
df_train = df_europarl[~df_europarl["year"].isin([2010, 2011])][["text","speaker_party"]]
df_train = pd.concat([df_train, filtered_debates], ignore_index=True)
df_train.drop_duplicates(subset=["text"], inplace=True)

In [64]:
df_train["text"].value_counts()

text
Ladies and gentlemen, once again, I should like to convey my heartfelt thanks for the trust you showed in me by electing me President of the European Parliament yesterday. To follow the various Presidents who have left their mark on the history of our Parliament ever since it was first elected by direct universal suffrage, and to follow you, José Maria Gil-Robles, who so successfully led Parliament along the path of democratic progress\nAlas, this building is not without its teething troubles, towards which we and the media must show indulgence. We are bearing the brunt of these but I can assure you that at the end of this week we will compile a detailed list of all the functional problems which have come to our attention and we will do everything possible to remedy the situation before the next part-session.\nI am aware of the daunting task which awaits us during the two and a half years of my Presidency. Our first duty, as I see it today, will be to demand full recognition of th

In [65]:
df_train.to_csv("data/combined/train.csv")
df_dev.to_csv("data/combined/dev.csv")
df_test.to_csv("data/combined/test.csv")